In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 680, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 680 (delta 23), reused 13 (delta 7), pack-reused 641 (from 2)
Receiving objects: 100% (680/680), 562.04 KiB | 3.45 MiB/s, done.
Resolving deltas: 100% (456/456), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [4]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50266, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154048, 41)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2975, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27820, 41)
Loaded: raw_syp_simas_sales_bills.csv -> (12944, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (37971, 38)
Loaded: raw_hq_icmas_products.csv -> (114956, 94)
Loaded: raw_hq_sidet_sales_lines.csv -> (733392, 38)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276005, 49)
Loaded: raw_hq_armas_receivable.csv -> (2227, 20)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)


In [5]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/statement"

In [6]:
!pip install xlrd

In [7]:
import os
import pandas as pd

data_statement = {}

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    ext = os.path.splitext(file)[1].lower()

    try:
        if ext == ".csv":
            # text file
            last_err = None
            for enc in ["utf-8-sig", "cp874", "cp1252", "latin1"]:
                try:
                    df = pd.read_csv(path, skiprows=10, encoding=enc, low_memory=False)
                    source = f"csv-{enc}"
                    break
                except Exception as e:
                    last_err = e
            else:
                raise last_err

        elif ext == ".xlsx":
            # modern Excel
            df = pd.read_excel(path, header=10, engine="openpyxl")
            source = "xlsx-openpyxl"

        elif ext == ".xls":
            # old Excel binary format
            df = pd.read_excel(path, header=10, engine="xlrd")
            source = "xls-xlrd"

        else:
            continue

        for col in ["BCODE", "ITEMNO", "BILLNO"]:
            if col in df.columns:
                df[col] = df[col].astype("string")

        key = os.path.splitext(file)[0]
        data_statement[key] = df
        print(f"Loaded: {file} -> {df.shape} ({source})")

    except Exception as e:
        print(f"Failed: {file} -> {type(e).__name__}: {e}")

Loaded: raw_statement_6184.xls -> (31, 9) (xls-xlrd)
Loaded: raw_statement_3557.csv -> (138, 13) (csv-utf-8-sig)
Loaded: raw_statement_7236.csv -> (92, 13) (csv-utf-8-sig)
Loaded: raw_statement_1139.xls -> (42, 9) (xls-xlrd)
Loaded: raw_statement_0393.csv -> (67, 13) (csv-utf-8-sig)


In [8]:
df_6184 = data_statement["raw_statement_6184"].copy()
df_1139 = data_statement["raw_statement_1139"].copy()

df_0393 = data_statement["raw_statement_0393"].copy()
df_3557 = data_statement["raw_statement_3557"].copy()
df_7236 = data_statement["raw_statement_7236"].copy()


In [11]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()
df_pimas = data["raw_syp_pimas_purchase_bills.csv"].copy()
df_pvmas = data["raw_hq_pvmas_notes_vouchers.csv"].copy()

In [14]:
import pandas as pd

def filter_last_1year(df, date_col="BILLDATE"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    cutoff = pd.Timestamp.today() - pd.DateOffset(years=1)

    return df[df[date_col] >= cutoff]

df_simas = filter_last_1year(df_simas)
df_pimas = filter_last_1year(df_pimas)


In [40]:
import pandas as pd

def match_billno_by_amount(
    df_source: pd.DataFrame,
    df_statement: pd.DataFrame,
    *,
    source_name: str,
    source_amount_col: str = "AMOUNT",
    statement_amount_col: str = "Amount",
    source_billno_col: str = "BILLNO",
    tolerance: float = 1.0,
    keep_source_amount: bool = True,
    keep_amount_diff: bool = True,
) -> pd.DataFrame:
    """
    Match df_statement[statement_amount_col] to df_source[source_amount_col]
    using nearest amount within +/- tolerance, and append source columns
    into df_statement with prefixed output names.

    Output columns:
      - {source_name}.BILLNO_MATCHED
      - {source_name}.SOURCE_AMOUNT
      - {source_name}.AMOUNT_DIFF
    """
    source = df_source.copy()
    stmt = df_statement.copy()

    source[source_amount_col] = pd.to_numeric(
        source[source_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    stmt[statement_amount_col] = pd.to_numeric(
        stmt[statement_amount_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    stmt["_row_id"] = range(len(stmt))

    source_valid = source.dropna(subset=[source_amount_col]).copy()
    stmt_valid = stmt.dropna(subset=[statement_amount_col]).copy()

    if source_valid.empty:
        out = stmt.copy()
        out[f"{source_name}.BILLNO_MATCHED"] = pd.NA
        if keep_source_amount:
            out[f"{source_name}.SOURCE_AMOUNT"] = pd.NA
        if keep_amount_diff:
            out[f"{source_name}.AMOUNT_DIFF"] = pd.NA
        return out.drop(columns=["_row_id"])

    source_valid = source_valid.sort_values(source_amount_col)
    stmt_valid = stmt_valid.sort_values(statement_amount_col)

    cols_to_keep = [source_amount_col, source_billno_col]
    source_for_merge = source_valid[cols_to_keep].copy()

    matched = pd.merge_asof(
        stmt_valid,
        source_for_merge,
        left_on=statement_amount_col,
        right_on=source_amount_col,
        direction="nearest",
        tolerance=tolerance,
    )

    billno_out_col = f"{source_name}.BILLNO_MATCHED"
    source_amount_out_col = f"{source_name}.SOURCE_AMOUNT"
    diff_out_col = f"{source_name}.AMOUNT_DIFF"

    matched = matched.rename(columns={
        source_billno_col: billno_out_col,
        source_amount_col: source_amount_out_col,
    })

    if keep_amount_diff:
        matched[diff_out_col] = (
            matched[statement_amount_col] - matched[source_amount_out_col]
        ).abs()

    keep_cols = ["_row_id", billno_out_col]
    if keep_source_amount:
        keep_cols.append(source_amount_out_col)
    if keep_amount_diff:
        keep_cols.append(diff_out_col)

    out = stmt.merge(
        matched[keep_cols],
        on="_row_id",
        how="left",
    )

    out = out.sort_values("_row_id").drop(columns=["_row_id"])
    return out


def match_multiple_sources_by_amount(
    df_statement: pd.DataFrame,
    source_configs: list[dict],
    *,
    statement_amount_col: str = "Amount",
    tolerance: float = 1.0,
    keep_source_amount: bool = True,
    keep_amount_diff: bool = True,
) -> pd.DataFrame:
    """
    Apply match_billno_by_amount repeatedly using a list of source configs.

    Each config should look like:
    {
        "df_source": df_simas,
        "source_name": "sales",
        "source_amount_col": "AFTERTAX",
        "source_billno_col": "BILLNO",   # optional
        "tolerance": 1.0,                # optional
    }
    """
    out = df_statement.copy()

    for cfg in source_configs:
        out = match_billno_by_amount(
            df_source=cfg["df_source"],
            df_statement=out,
            source_name=cfg["source_name"],
            source_amount_col=cfg.get("source_amount_col", "AMOUNT"),
            statement_amount_col=cfg.get("statement_amount_col", statement_amount_col),
            source_billno_col=cfg.get("source_billno_col", "BILLNO"),
            tolerance=cfg.get("tolerance", tolerance),
            keep_source_amount=cfg.get("keep_source_amount", keep_source_amount),
            keep_amount_diff=cfg.get("keep_amount_diff", keep_amount_diff),
        )

    return out

In [43]:
source_configs = [
    {
        "df_source": df_simas,
        "source_name": "sales",
        "source_amount_col": "AFTERTAX",
    },
    {
        "df_source": df_pimas,
        "source_name": "purchases",
        "source_amount_col": "AFTERTAX",
    },
    {
        "df_source": df_pvmas,
        "source_name": "vouchers",
        "source_amount_col": "BILLAMT",
        "source_billno_col": "VOUCNO",
    },
]

df_out_6184 = match_multiple_sources_by_amount(
    df_statement=df_6184,
    source_configs=source_configs,
    tolerance=1.0,
)

df_out_1139 = match_multiple_sources_by_amount(
    df_statement=df_1139,
    source_configs=source_configs,
    tolerance=1.0,
)



In [46]:
df_out_1139.columns

Index(['Date', 'Teller Id', 'Transaction Code', 'Description', 'Cheque No.',
       'Amount', 'Tax', 'Balance', 'Init Br.', 'sales.BILLNO_MATCHED',
       'sales.SOURCE_AMOUNT', 'sales.AMOUNT_DIFF', 'purchases.BILLNO_MATCHED',
       'purchases.SOURCE_AMOUNT', 'purchases.AMOUNT_DIFF',
       'vouchers.BILLNO_MATCHED', 'vouchers.SOURCE_AMOUNT',
       'vouchers.AMOUNT_DIFF'],
      dtype='object')

In [47]:
df_out_1139[['Date', 'Description', 'Cheque No.',
       'Amount', 'Tax', 'Balance', 'Init Br.', 'sales.BILLNO_MATCHED', 'sales.AMOUNT_DIFF', 'purchases.BILLNO_MATCHED'
       , 'purchases.AMOUNT_DIFF',
       'vouchers.BILLNO_MATCHED',
       'vouchers.AMOUNT_DIFF']]

,Date,Description,Cheque No.,Amount,Tax,Balance,Init Br.,sales.BILLNO_MATCHED,sales.AMOUNT_DIFF,purchases.BILLNO_MATCHED,purchases.AMOUNT_DIFF,vouchers.BILLNO_MATCHED,vouchers.AMOUNT_DIFF
0,03/02/2026 08:00:33,AirPay(TH)/ช้อปปี้เพย์ (ประเทศไทย)/20119,NaN,45847.00,0,47855.50,108682,<NA>,NaN,<NA>,NaN,P6602-031,0.76
1,03/02/2026 08:00:33,AirPay(TH)/ช้อปปี้เพย์ (ประเทศไทย)/20119,NaN,7975.00,0,55830.50,108682,<NA>,NaN,<NA>,NaN,NaN,NaN
2,03/02/2026 09:00:14,AirPay(TH)/ช้อปปี้เพย์ (ประเทศไทย)/20119,NaN,5378.00,0,61208.50,108682,<NA>,NaN,<NA>,NaN,NaN,NaN
3,04/02/2026 02:08:25,024-6993647915 Future Amount: 12733.49,NaN,12733.49,0,73941.99,248,<NA>,NaN,<NA>,NaN,NaN,NaN
4,04/02/2026 03:00:27,BPS/017/01/Lazada Ltd./108682,NaN,47193.12,0,121135.11,108682,<NA>,NaN,<NA>,NaN,NaN,NaN
5,04/02/2026 03:00:28,BPS/017/01/Lazada Ltd./108682,NaN,74844.45,0,195979.56,108682,<NA>,NaN,<NA>,NaN,NaN,NaN
6,04/02/2026 03:00:38,BPS/017/01/Lazada Ltd./108682,NaN,367.12,0,196346.68,108682,8K68-0024462,0.12,<NA>,NaN,NaN,0.88
7,04/02/2026 03:01:04,BPS/017/01/Lazada Ltd./108682,NaN,2971.18,0,199317.86,108682,<NA>,NaN,<NA>,NaN,P59-00077,0.21
8,04/02/2026 18:32:08,TR to 2486006184 KIATCHAI AUTO PART 2007,NaN,-190000.00,0,9317.86,248,<NA>,NaN,<NA>,NaN,NaN,NaN
9,10/02/2026 08:00:43,AirPay(TH)/ช้อปปี้เพย์ (ประเทศไทย)/20119,NaN,22268.00,0,31585.86,108682,<NA>,NaN,<NA>,NaN,NaN,NaN


In [12]:
df_6184.head()

,Date,Teller Id,Transaction Code,Description,Cheque No.,Amount,Tax,Balance,Init Br.
0,04/02/2026 18:32:08,ITBANK,PBDDT,TR fr 2480421139 KIATCHAI AUTO PART 2007,NaN,190000.0,0,703701.57,248
1,04/02/2026 18:34:57,ITBANK,IORDWT,TR to 3253014733 WEISUNTHAI 202602040050,NaN,-10709.0,0,692992.57,248
2,05/02/2026 18:22:54,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127713.0,-26036.0,0,666956.57,700
3,06/02/2026 18:15:43,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127716.0,-18050.0,0,648906.57,700
4,09/02/2026 19:11:24,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127717.0,-5960.0,0,642946.57,700


In [57]:
df = df_7236

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [58]:
df

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,วันที่,เวลา/ วันที่มีผล,รายการ,ถอนเงิน,ฝากเงิน,ยอดคงเหลือ,ช่องทาง,รายละเอียด
0,01-02-26,NaN,ยอดยกมา,NaN,NaN,"4,191.89",NaN,NaN
1,01-02-26,12:05,รับโอนเงิน,NaN,"25,120.00","29,311.89",Internet/Mobile SCB,จาก SCB X3875 นางสาว ธัญญพัทธ์ ท++
2,02-02-26,09:35,รับโอนเงิน,NaN,"62,506.50","91,818.39",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
3,02-02-26,09:37,โอนเงิน,"90,000.00",NaN,"1,818.39",K BIZ,โอนไป X3557 บจก. เกียรติชัยอะไ++
4,02-02-26,17:58,รับโอนเงิน,NaN,"141,552.50","143,370.89",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
...,...,...,...,...,...,...,...,...
87,28-02-26,11:52,รับโอนเงิน,NaN,"54,444.01","67,230.29",Internet/Mobile KTB,จาก KTB X3316 AUGKANA WILAINIKH++
88,28-02-26,15:34,รับโอนเงิน,NaN,"67,340.50","134,570.79",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
89,28-02-26,15:37,รับโอนเงิน,NaN,"69,815.00","204,385.79",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
90,28-02-26,15:49,โอนเงิน,"200,000.00",NaN,"4,385.79",K BIZ,โอนไป X3557 บจก. เกียรติชัยอะไ++
